In [21]:
import os
import html

import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt
import contextily as ctx

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

In [23]:
clustered

,lpa_name,decision_date,bo_system,valid_date,description,lapsed_date,centroid,lpa_app_no,status,application_type,decision,application_details_site_area,is_full_planning,geometry_point,is_possible_child,is_parent,is_possible_child_cluster,geometry
5,Ealing,NaT,CSV Data Import,03/04/2025,Hybrid planning application for a four-phased ...,None,"{'lat': 51.512324, 'lon': -0.394453}",251348CONS,Application Under Consideration,All Other,None,NaN,True,POINT (-0.394453 51.512324),True,True,4,POINT (-0.394 51.512)
10,Havering,2021-01-14,CSV Data Import,05/05/2020,New data storage development consisting of new...,14/01/2024,"{'lat': '51.585996', 'lon': '0.1693495'}",P0614.20,Approved,All Other,Approved,1.598,True,POINT (0.1693495 51.585996),True,True,6,"POLYGON ((550439.207 189785.6, 550480.389 1897..."
15,Hillingdon,NaT,CSV Data Import,28/03/2025,Hybrid planning application for a four-phased ...,None,"{'lon': -0.394453, 'lat': 51.512324}",78343/APP/2025/719,Application Under Consideration,All Other,Approved,1.800,True,POINT (-0.394453 51.512324),True,True,4,POINT (-0.394 51.512)
16,Hounslow,2019-10-15,Portal,23/08/2019,Erection of new gantry to accommodate addition...,15/10/2022,"{'lon': '-0.4542407', 'lat': '51.4466651'}",P/2019/3115,Approved,All Other,Approved,0.391,True,POINT (-0.4542407 51.4466651),True,True,8,"POLYGON ((507498.24 173122.98, 507506.97 17313..."
17,Hounslow,NaT,Northgate M3,19/03/2025,"Site clearance and preparation, including demo...",None,"{'lon': -7.55716, 'lat': 49.766807}",C/2025/0873,Application Received,All Other,None,NaN,True,POINT (-7.55716 49.766807),True,True,9,"POLYGON ((516336.219 173039.727, 516339.919 17..."
19,Hounslow,2000-11-09,Portal,None,Installation of mezzanine floors and erection ...,09/11/2005,"{'lon': '-0.4541577', 'lat': '51.446565'}",P/2000/1917,Completed,All Other,Approved,0.942,True,POINT (-0.4541577 51.446565),True,True,8,"POLYGON ((507528.637 173148.113, 507530.754 17..."
20,Hounslow,2025-04-29,Northgate M3,25/04/2025,Redevelopment of site to deliver extension to ...,29/04/2028,"{'lon': -0.327669, 'lat': 51.444245}",C/2025/1325,Unknown,All Other,No Objection to Proposal (OBS only),NaN,True,POINT (-0.327669 51.444245),True,True,9,"POLYGON ((516336.219 173039.727, 516339.919 17..."
21,Islington,2022-02-25,Portal,16/06/2021,"Change of use of parts lower ground, ground, f...",25/02/2025,"{'lon': '-0.0864751', 'lat': '51.5231098'}",P2021/1761/FUL,Completed,All Other,Approved,0.015,True,POINT (-0.0864751 51.5231098),True,True,10,"POLYGON ((532852.793 182193.947, 532843.295 18..."
22,Islington,2015-07-21,Agile APAS,26/05/2015,"Temporary change of use of basement, ground an...",21/07/2018,"{'lat': '51.5231098', 'lon': '-0.0864751'}",P2015/2114/FUL,Insufficient Fee,All Other,Approved,0.015,True,POINT (-0.0864751 51.5231098),True,True,10,"POLYGON ((532852.793 182193.947, 532843.295 18..."
24,Islington,2013-11-25,Agile APAS,01/08/2013,The replacement of 10 existing cooling towers ...,25/11/2016,"{'lat': '51.5192439', 'lon': '-0.1042515'}",P2013/2803/LBC,Insufficient Fee,All Other,Approved,0.303,True,POINT (-0.1042515 51.5192439),True,True,12,"POLYGON ((531648.742 181751.094, 531650.647 18..."


In [22]:
# -----------------------
# Files
# -----------------------
data_file = "validation_is_true_parent_source.gpkg"
label_file = "validation_is_true_parent_output.csv"
layer_name = "validation_is_true_parent"   # your layer name inside the gpkg

# -----------------------
# Load source data (GeoPackage is EPSG:27700)
# -----------------------
gdf = gpd.read_file(data_file, layer=layer_name).copy()

# Ensure CRS is set correctly
if gdf.crs is None:
    gdf = gdf.set_crs("EPSG:27700")
else:
    # If it's wrong/missing in the file, force it here (only if you are sure)
    # gdf = gdf.set_crs("EPSG:27700", allow_override=True)
    pass

# Ensure decision_date is datetime
if "decision_date" in gdf.columns:
    gdf["decision_date"] = pd.to_datetime(
        gdf["decision_date"],
        errors="coerce",
        dayfirst=True
    )


# Ensure IDs are strings
gdf["lpa_app_no"] = gdf["lpa_app_no"].astype(str)

# Must have geometry + cluster column
if "geometry" not in gdf.columns:
    raise ValueError("Expected a 'geometry' column in the source file.")
if "is_possible_child_cluster" not in gdf.columns:
    raise ValueError("Expected column 'is_possible_child_cluster' in source data.")

# Keep only rows with geometry
gdf = gdf[gdf.geometry.notna()].copy()

# Only clustered rows (exclude NA clusters)
clustered = gdf[gdf["is_possible_child_cluster"].notna()].copy()
clustered["is_possible_child_cluster"] = clustered["is_possible_child_cluster"].astype(int)

# -----------------------
# Create output file if missing
# Output must have: lpa_app_no, description, is_parent
# -----------------------
if not os.path.exists(label_file):
    pd.DataFrame({"lpa_app_no": [], "description": [], "is_parent": []}).to_csv(label_file, index=False)

# Load existing validations
validated = pd.read_csv(label_file)
validated_set = set(validated["lpa_app_no"].dropna().astype(str))

# Remove already validated
clustered = clustered[~clustered["lpa_app_no"].isin(validated_set)].copy()

# Build clusters list
cluster_ids = sorted(clustered["is_possible_child_cluster"].unique().tolist())


# -----------------------
# Helpers
# -----------------------
def append_validations(rows_to_write):
    """Append multiple validated rows to CSV immediately."""
    out_df = pd.DataFrame(rows_to_write, columns=["lpa_app_no", "description", "is_parent"])
    out_df.to_csv(label_file, mode="a", header=False, index=False)


def _to_web_mercator(gdf_to_plot):
    # contextily basemaps expect EPSG:3857
    if gdf_to_plot.crs is None:
        gdf_to_plot = gdf_to_plot.set_crs("EPSG:27700", allow_override=True)
    return gdf_to_plot.to_crs(epsg=3857)


def _safe_str(v):
    """Stringify values safely for HTML (prevents blanks / invisible content)."""
    if v is None:
        return ""
    # pandas NA / NaT
    try:
        if pd.isna(v):
            return ""
    except Exception:
        pass
    # timestamps
    if isinstance(v, pd.Timestamp):
        if pd.isna(v):
            return ""
        return v.strftime("%Y-%m-%d")
    return str(v)


def _render_map(one_row_gdf):
    """Render a small map for a single geometry row with basemap."""
    to_plot_web = _to_web_mercator(one_row_gdf)

    fig, ax = plt.subplots(figsize=(4.6, 4.6))
    # plot boundary + fill (boundary only is OK too)
    to_plot_web.boundary.plot(ax=ax)

    # bounds (guard against weird/empty bounds)
    minx, miny, maxx, maxy = to_plot_web.total_bounds
    if not all(pd.notna([minx, miny, maxx, maxy])) or (minx == maxx) or (miny == maxy):
        # fallback: let geopandas set extent
        ax.autoscale()
    else:
        pad_x = (maxx - minx) * 0.2
        pad_y = (maxy - miny) * 0.2
        ax.set_xlim(minx - pad_x, maxx + pad_x)
        ax.set_ylim(miny - pad_y, maxy + pad_y)

    ctx.add_basemap(
        ax,
        crs=to_plot_web.crs,
        source=ctx.providers.CartoDB.Positron,
        zoom=18
    )

    ax.set_axis_off()
    plt.show()


def _row_card_html(row_dict):
    """
    Middle panel: only show:
      - decision_date
      - application_type_full
      - status
      - description (scroll box)
    """
    decision_date = html.escape(_safe_str(row_dict.get("decision_date", "")))
    app_type = html.escape(_safe_str(row_dict.get("application_type_full", "")))
    status = html.escape(_safe_str(row_dict.get("status", "")))

    desc_raw = _safe_str(row_dict.get("description", ""))
    # preserve newlines & escape HTML
    desc = html.escape(desc_raw).replace("\n", "<br>")

    return f"""
    <div style="border:1px solid #ddd; padding:8px; border-radius:6px;">
      <div><b>Decision date:</b> {decision_date}</div>
      <div><b>Application type:</b> {app_type}</div>
      <div><b>Status:</b> {status}</div>
      <div style="margin-top:8px;"><b>Description:</b></div>
      <div style="max-height:240px; overflow:auto; border:1px solid #ccc; padding:8px; border-radius:6px; white-space:normal;">
        {desc}
      </div>
    </div>
    """


def _selection_status_html(cluster_rows, decisions):
    lines = []
    for _, r in cluster_rows.iterrows():
        app = str(r["lpa_app_no"])
        status = decisions.get(app, None)
        if status is True:
            s = "✅ parent"
        elif status is False:
            s = "🧒 child"
        else:
            s = "⬜ not set"
        lines.append(f"<li><b>{html.escape(app)}</b>: {s}</li>")
    return "<ul style='margin:6px 0 0 18px;'>" + "".join(lines) + "</ul>"


def _all_set(cluster_rows, decisions):
    return all(str(r["lpa_app_no"]) in decisions for _, r in cluster_rows.iterrows())


# -----------------------
# Widgets + state
# -----------------------
output = widgets.Output()
header = widgets.HTML()

next_cluster_btn = widgets.Button(description="Next cluster ▶", button_style="info", disabled=True)
skip_cluster_btn = widgets.Button(description="Skip cluster ⏭️", button_style="warning")

cluster_state = {
    "cluster_idx": 0,
    "decisions": {}  # {lpa_app_no: True/False}
}


def show_cluster():
    with output:
        clear_output()

        if cluster_state["cluster_idx"] >= len(cluster_ids):
            header.value = "<h3>🎉 All clusters have been validated (or skipped).</h3>"
            next_cluster_btn.disabled = True
            skip_cluster_btn.disabled = True
            return

        cid = cluster_ids[cluster_state["cluster_idx"]]
        cluster_rows = clustered[clustered["is_possible_child_cluster"] == cid].copy()

        decisions = cluster_state["decisions"]
        header.value = f"""
        <h3>Cluster {cluster_state['cluster_idx']+1} / {len(cluster_ids)} — ID: {cid}</h3>
        <div>Items in cluster: <b>{len(cluster_rows)}</b></div>
        <div style="margin-top:6px;"><b>Selections:</b>{_selection_status_html(cluster_rows, decisions)}</div>
        """

        # enable/disable Next cluster button
        next_cluster_btn.disabled = not _all_set(cluster_rows, decisions)

        for _, r in cluster_rows.iterrows():
            app_no = str(r["lpa_app_no"])

            keep_parent_btn = widgets.Button(description="Keep as parent", button_style="success")
            keep_child_btn = widgets.Button(description="Keep as child", button_style="danger")

            def _make_onclick(app_no, is_parent_value):
                def _cb(_):
                    cluster_state["decisions"][app_no] = is_parent_value
                    show_cluster()
                return _cb

            keep_parent_btn.on_click(_make_onclick(app_no, True))
            keep_child_btn.on_click(_make_onclick(app_no, False))

            map_out = widgets.Output(layout=widgets.Layout(width="34%"))
            info_html = widgets.HTML(value=_row_card_html(r.to_dict()), layout=widgets.Layout(width="46%"))
            btn_box = widgets.VBox([keep_parent_btn, keep_child_btn], layout=widgets.Layout(width="20%"))

            row_box = widgets.HBox([map_out, info_html, btn_box])

            display(HTML(f"<hr><div><b>Application:</b> {html.escape(app_no)}</div>"))
            display(row_box)

            # draw map
            with map_out:
                clear_output()
                one = gpd.GeoDataFrame([r], geometry="geometry", crs=gdf.crs)
                _render_map(one)


def on_next_cluster(_):
    cid = cluster_ids[cluster_state["cluster_idx"]]
    cluster_rows = clustered[clustered["is_possible_child_cluster"] == cid].copy()

    decisions = cluster_state["decisions"]
    if not _all_set(cluster_rows, decisions):
        return  # safety; button should be disabled anyway

    rows_to_write = []
    for _, r in cluster_rows.iterrows():
        app_no = str(r["lpa_app_no"])
        rows_to_write.append({
            "lpa_app_no": app_no,
            "description": _safe_str(r.get("description", "")),
            "is_parent": decisions[app_no]
        })

    append_validations(rows_to_write)

    # advance
    cluster_state["cluster_idx"] += 1
    cluster_state["decisions"] = {}
    show_cluster()


def on_skip_cluster(_):
    cluster_state["cluster_idx"] += 1
    cluster_state["decisions"] = {}
    show_cluster()


next_cluster_btn.on_click(on_next_cluster)
skip_cluster_btn.on_click(on_skip_cluster)

# -----------------------
# Display UI
# -----------------------
display(header)
display(widgets.HBox([next_cluster_btn, skip_cluster_btn]))
display(output)

show_cluster()

HTML(value='')

Output()